In [61]:
#%pip install pyarrow
%pip install tqdm

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
Note: you may need to restart the kernel to use updated packages.


In [59]:
from pathlib import Path

# Rechercher la racine du projet en remontant depuis le dossier actuel
candidats = [Path.cwd(), *Path.cwd().parents]

RACINE_PROJET = next(
    (
        dossier for dossier in candidats
        if (dossier / "data").is_dir()
        and (dossier / "notebooks").is_dir()
    ),
    None
)

if RACINE_PROJET is None:
    raise FileNotFoundError("Impossible de trouver la racine du projet COFI.")

print("Racine du projet :", RACINE_PROJET)
print("Dossier des données brutes :", RACINE_PROJET / "data" / "raw")
print("Dossier des données traitées :", RACINE_PROJET / "data" / "processed")

Racine du projet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI
Dossier des données brutes : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/raw
Dossier des données traitées : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed


In [15]:
import pandas as pd
from pathlib import Path

# Chemin vers le fichier Parquet
fichier = Path("../data/raw/multilingual_gec_fr.parquet")

# Vérifier que le fichier existe
print("Fichier trouvé :", fichier.exists())

# Charger le corpus
df = pd.read_parquet(fichier)

# Afficher les informations principales
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])
print("\nColonnes disponibles :")
print(df.columns.tolist())

# Afficher les 10 premières lignes
display(df.head(10))

Fichier trouvé : True
Nombre de lignes : 66500
Nombre de colonnes : 6

Colonnes disponibles :
['lang', 'sentence', 'modified', 'transformation', 'sec_transformation', '__index_level_0__']


,lang,sentence,modified,transformation,sec_transformation,__index_level_0__
0,fr,Il est très important de parler une langue étr...,fix grammar: Il est très importante de parler ...,AdjectiveGenderChangeDestroyer,NaN,44864
1,fr,"En 1933-1934, elle plaide auprès du président ...","fix grammar: En 1933-1934, elle plaide auprès ...",GenderDeterminerDestroyer,NaN,54700
2,fr,La lune était observée par cette porte afin d’...,fix grammar: Les lune était observée par cette...,PluralDeterminerDestroyer,NaN,53121
3,fr,Il finit sa saison 2017-2018 avec une moyenne ...,fix grammar: Il finissent sa saison 2017-2018 ...,FrVerbNumberChanger,NaN,5211
4,fr,"Le général fatimide envahit l'émirat, vainc un...",fix grammar: Le fatimide général envahit l'émi...,AdjectiveMisplacerDestroyer,NaN,36043
5,fr,"Le chapiteau est d'un style dorique, cependant...",fix grammar: Le chapiteau est d'un style doriq...,AdjectiveGenderChangeDestroyer,NaN,31321
6,fr,Ce n'est pas une bonne voiture mais c'est touj...,fix grammar: Ce n'est pas une bonne voiture ma...,PluralSingularNounDestroyer,NaN,4715
7,fr,Joseph-Cléophas Saindon fut le fondateur de la...,fix grammar: Joseph-Cléophas Saindon fut la fo...,GenderDeterminerDestroyer,NaN,44414
8,fr,"Les propriétaires de la ferme Walserhof, la fa...",fix grammar: Le propriétaires du la ferme Wals...,ContractionsDestroyer,PluralDeterminerDestroyer,7165
9,fr,C'est un de mes points forts.,fix grammar: C'est un du mes forts points.,AdjectiveMisplacerDestroyer,ContractionsDestroyer,97


In [16]:
# Afficher 5 paires de phrases sans troncature
for i, ligne in df.head(5).iterrows():
    print(f"\nEXEMPLE {i + 1}")
    print("Type de faute :", ligne["transformation"])
    print("Phrase incorrecte :", ligne["modified"])
    print("Phrase correcte   :", ligne["sentence"])
    print("-" * 80)


EXEMPLE 1
Type de faute : AdjectiveGenderChangeDestroyer
Phrase incorrecte : fix grammar: Il est très importante de parler une langue étrangère.
Phrase correcte   : Il est très important de parler une langue étrangère.
--------------------------------------------------------------------------------

EXEMPLE 2
Type de faute : GenderDeterminerDestroyer
Phrase incorrecte : fix grammar: En 1933-1934, elle plaide auprès du président pour un adhésion des États-Unis à l'
Phrase correcte   : En 1933-1934, elle plaide auprès du président pour une adhésion des États-Unis à l'
--------------------------------------------------------------------------------

EXEMPLE 3
Type de faute : PluralDeterminerDestroyer
Phrase incorrecte : fix grammar: Les lune était observée par cette porte afin d’annoncer les début du mois Hégirien), et Bab El Khmer.
Phrase correcte   : La lune était observée par cette porte afin d’annoncer le début du mois Hégirien), et Bab El Khmer.
-------------------------------------

In [17]:
print("RÉPARTITION DES LANGUES")
print(df["lang"].value_counts())

print("\nPRÉFIXE DES PHRASES INCORRECTES")
print(
    df["modified"]
    .str.startswith("fix grammar: ")
    .value_counts()
)

print("\nVALEURS MANQUANTES")
print(df.isna().sum())

print("\nPAIRES DUPLIQUÉES")
print(df.duplicated(subset=["modified", "sentence"]).sum())

print("\nPHRASES IDENTIQUES ENTRE LES DEUX COLONNES")
print((df["modified"] == df["sentence"]).sum())

RÉPARTITION DES LANGUES
lang
fr    66500
Name: count, dtype: int64

PRÉFIXE DES PHRASES INCORRECTES
modified
True    66500
Name: count, dtype: int64

VALEURS MANQUANTES
lang                      0
sentence                  0
modified                  0
transformation           86
sec_transformation    57522
__index_level_0__         0
dtype: int64

PAIRES DUPLIQUÉES
638

PHRASES IDENTIQUES ENTRE LES DEUX COLONNES
0


In [18]:
# Rechercher les phrases qui se terminent par une apostrophe
phrases_tronquees = df[
    df["sentence"].str.rstrip().str.endswith("'")
]

print("Phrases correctes se terminant par une apostrophe :",
      len(phrases_tronquees))

print("\nQuelques exemples :")

for _, ligne in phrases_tronquees.head(10).iterrows():
    print("Incorrecte :", ligne["modified"])
    print("Correcte   :", ligne["sentence"])
    print("-" * 80)

Phrases correctes se terminant par une apostrophe : 2470

Quelques exemples :
Incorrecte : fix grammar: En 1933-1934, elle plaide auprès du président pour un adhésion des États-Unis à l'
Correcte   : En 1933-1934, elle plaide auprès du président pour une adhésion des États-Unis à l'
--------------------------------------------------------------------------------
Incorrecte : fix grammar: Le fatimide général envahit l'émirat, vainc une byzantine force dirigée par le dux d'
Correcte   : Le général fatimide envahit l'émirat, vainc une force byzantine dirigée par le dux d'
--------------------------------------------------------------------------------
Incorrecte : fix grammar: L'situation intérieure de l'double monarchie, les relations tendues entre l'
Correcte   : La situation intérieure de la double monarchie, les relations tendues entre l'
--------------------------------------------------------------------------------
Incorrecte : fix grammar: Il semble que Cheissoux ait été une étape

In [19]:
# Calculer la longueur des phrases
df["longueur_correcte"] = df["sentence"].str.len()
df["longueur_incorrecte"] = df["modified"].str.len()

print("Longueur minimale :", df["longueur_correcte"].min())
print("Longueur maximale :", df["longueur_correcte"].max())
print("Longueur moyenne :", round(df["longueur_correcte"].mean(), 2))

print("\nPhrases correctes de moins de 10 caractères :",
      (df["longueur_correcte"] < 10).sum())

print("Phrases correctes de plus de 300 caractères :",
      (df["longueur_correcte"] > 300).sum())

Longueur minimale : 5
Longueur maximale : 255
Longueur moyenne : 88.44

Phrases correctes de moins de 10 caractères : 40
Phrases correctes de plus de 300 caractères : 0


In [20]:
import re

# Motifs de fins de phrases à examiner
motifs = {
    "apostrophe": r"['’]$",
    "préposition": r"\b(de|à|dans|pour|avec|sur|par|en|vers)$",
    "conjonction": r"\b(et|ou|mais|car|que|qui|dont)$",
    "déterminant": r"\b(le|la|les|un|une|des|du|ce|cette|ces)$"
}

for nom, motif in motifs.items():
    nombre = df["sentence"].str.rstrip().str.contains(
        motif, case=False, regex=True
    ).sum()

    print(f"{nom.capitalize()} : {nombre}")

Apostrophe : 2877


/tmp/ipykernel_104492/2081205870.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nombre = df["sentence"].str.rstrip().str.contains(


Préposition : 0
Conjonction : 0
Déterminant : 0


In [21]:
phrases_courtes = df[df["longueur_correcte"] < 10]

for _, ligne in phrases_courtes.iterrows():
    print("Incorrecte :", ligne["modified"])
    print("Correcte   :", ligne["sentence"])
    print("-" * 60)

Incorrecte : fix grammar: Abruti 
Correcte   : Abruti !
------------------------------------------------------------
Incorrecte : fix grammar: Je pue
Correcte   : Je pue.
------------------------------------------------------------
Incorrecte : fix grammar: Sauve-les.
Correcte   : Sauve-la.
------------------------------------------------------------
Incorrecte : fix grammar: oMntez !
Correcte   : Montez !
------------------------------------------------------------
Incorrecte : fix grammar: On y vais 
Correcte   : On y va ?
------------------------------------------------------------
Incorrecte : fix grammar: Bonjour.
Correcte   : Bonjour.
------------------------------------------------------------
Incorrecte : fix grammar: sauvela
Correcte   : Sauve-la.
------------------------------------------------------------
Incorrecte : fix grammar: Du meme.
Correcte   : De même.
------------------------------------------------------------
Incorrecte : fix grammar: Poilce !
Correcte   : Police

In [22]:
# Copie temporaire pour les vérifications
controle = df[["modified", "sentence"]].copy()

# Retirer le préfixe et les espaces au début et à la fin
controle["incorrect"] = (
    controle["modified"]
    .str.replace(r"^fix grammar:\s*", "", regex=True)
    .str.strip()
)

controle["correct"] = controle["sentence"].str.strip()

# Compter les paires identiques après nettoyage
identiques = controle[
    controle["incorrect"] == controle["correct"]
]

print("Paires identiques après retrait du préfixe :", len(identiques))

print("\nExemples de paires identiques :")
display(identiques.head(10))

# Compter les doublons exacts après retrait du préfixe
doublons = controle.duplicated(
    subset=["incorrect", "correct"]
).sum()

print("\nPaires dupliquées après retrait du préfixe :", doublons)

# Compter les phrases incorrectes vides
vides = (controle["incorrect"] == "").sum()

print("\nPhrases incorrectes vides :", vides)

# Compter les phrases correctes vides
vides_correctes = (controle["correct"] == "").sum()

print("Phrases correctes vides :", vides_correctes)

Paires identiques après retrait du préfixe : 88

Exemples de paires identiques :


,modified,sentence,incorrect,correct
728,fix grammar: Tom doit faire cela.,Tom doit faire cela.,Tom doit faire cela.,Tom doit faire cela.
781,fix grammar: Cela signifie que l',Cela signifie que l',Cela signifie que l',Cela signifie que l'
2070,fix grammar: Tom fait beaucoup d'erreurs.,Tom fait beaucoup d'erreurs.,Tom fait beaucoup d'erreurs.,Tom fait beaucoup d'erreurs.
3597,fix grammar: Aimeriez-vous venir avec nous ?,Aimeriez-vous venir avec nous ?,Aimeriez-vous venir avec nous ?,Aimeriez-vous venir avec nous ?
4292,fix grammar: D'accord ?,D'accord ?,D'accord ?,D'accord ?
4793,fix grammar: Soyez tolérants.,Soyez tolérants.,Soyez tolérants.,Soyez tolérants.
5768,fix grammar: Vas-tu voter pour Tom Jackson ?,Vas-tu voter pour Tom Jackson ?,Vas-tu voter pour Tom Jackson ?,Vas-tu voter pour Tom Jackson ?
6003,fix grammar: Penses-tu que Tom va mourir ?,Penses-tu que Tom va mourir ?,Penses-tu que Tom va mourir ?,Penses-tu que Tom va mourir ?
7041,fix grammar: Il reste peu d'argent.,Il reste peu d'argent.,Il reste peu d'argent.,Il reste peu d'argent.
7469,fix grammar: Aimeriez-vous être riches ?,Aimeriez-vous être riches ?,Aimeriez-vous être riches ?,Aimeriez-vous être riches ?



Paires dupliquées après retrait du préfixe : 639

Phrases incorrectes vides : 0
Phrases correctes vides : 0


In [23]:
import pandas as pd
from pathlib import Path

# 1. Copier le corpus original
corpus = df.copy()

# 2. Retirer le préfixe technique
corpus["incorrect"] = (
    corpus["modified"]
    .str.replace(r"^fix grammar:\s*", "", regex=True)
    .str.strip()
)

# 3. Nettoyer les espaces autour des phrases correctes
corpus["correct"] = corpus["sentence"].str.strip()

# 4. Retirer les paires vides
corpus = corpus[
    (corpus["incorrect"] != "") &
    (corpus["correct"] != "")
].copy()

# 5. Retirer les paires identiques
corpus = corpus[
    corpus["incorrect"] != corpus["correct"]
].copy()

# 6. Supprimer les doublons de paires
corpus = corpus.drop_duplicates(
    subset=["incorrect", "correct"]
).copy()

# 7. Ajouter la provenance
corpus["source"] = "huggingface_multilingual_gec"

# 8. Conserver uniquement les colonnes utiles
corpus = corpus[
    [
        "incorrect",
        "correct",
        "transformation",
        "sec_transformation",
        "source"
    ]
].reset_index(drop=True)

# 9. Afficher le bilan
print("Nombre de paires après nettoyage :", len(corpus))
print("\nColonnes :", corpus.columns.tolist())

display(corpus.head(10))

Nombre de paires après nettoyage : 65773

Colonnes : ['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']


,incorrect,correct,transformation,sec_transformation,source
0,Il est très importante de parler une langue ét...,Il est très important de parler une langue étr...,AdjectiveGenderChangeDestroyer,NaN,huggingface_multilingual_gec
1,"En 1933-1934, elle plaide auprès du président ...","En 1933-1934, elle plaide auprès du président ...",GenderDeterminerDestroyer,NaN,huggingface_multilingual_gec
2,Les lune était observée par cette porte afin d...,La lune était observée par cette porte afin d’...,PluralDeterminerDestroyer,NaN,huggingface_multilingual_gec
3,Il finissent sa saison 2017-2018 avec une moye...,Il finit sa saison 2017-2018 avec une moyenne ...,FrVerbNumberChanger,NaN,huggingface_multilingual_gec
4,"Le fatimide général envahit l'émirat, vainc un...","Le général fatimide envahit l'émirat, vainc un...",AdjectiveMisplacerDestroyer,NaN,huggingface_multilingual_gec
5,"Le chapiteau est d'un style dorique, cependant...","Le chapiteau est d'un style dorique, cependant...",AdjectiveGenderChangeDestroyer,NaN,huggingface_multilingual_gec
6,Ce n'est pas une bonne voiture mais c'est touj...,Ce n'est pas une bonne voiture mais c'est touj...,PluralSingularNounDestroyer,NaN,huggingface_multilingual_gec
7,Joseph-Cléophas Saindon fut la fondateur de le...,Joseph-Cléophas Saindon fut le fondateur de la...,GenderDeterminerDestroyer,NaN,huggingface_multilingual_gec
8,"Le propriétaires du la ferme Walserhof, la fam...","Les propriétaires de la ferme Walserhof, la fa...",ContractionsDestroyer,PluralDeterminerDestroyer,huggingface_multilingual_gec
9,C'est un du mes forts points.,C'est un de mes points forts.,AdjectiveMisplacerDestroyer,ContractionsDestroyer,huggingface_multilingual_gec


In [24]:
print("Paires identiques :",
      (corpus["incorrect"] == corpus["correct"]).sum())

print("Doublons :",
      corpus.duplicated(
          subset=["incorrect", "correct"]
      ).sum())

print("Valeurs manquantes :")
print(corpus.isna().sum())

Paires identiques : 0
Doublons : 0
Valeurs manquantes :
incorrect                 0
correct                   0
transformation            0
sec_transformation    56795
source                    0
dtype: int64


In [25]:
print("Nombre final de paires :", len(corpus))

print("\nRépartition des transformations :")
print(corpus["transformation"].value_counts().head(20))

print("\nRépartition des sources :")
print(corpus["source"].value_counts())

Nombre final de paires : 65773

Répartition des transformations :
transformation
PluralSingularNounDestroyer       7451
ApostropheChangerDestroyer        7318
AdjectiveMisplacerDestroyer       6937
ContractionsDestroyer             6470
GenderDeterminerDestroyer         5856
PluralDeterminerDestroyer         5186
PrepositionReplacerDestroyer      4310
PersonVerbDisagreement            4052
RemoveAccentsDestroyer            3894
FrRemovePunctuationDestroyer      3438
FrVerbNumberChanger               3211
PossessiveReplacerDestroyer       2102
ReflexivePronounRemover           1608
PasRemoverDestroyer               1365
AdjectiveGenderChangeDestroyer    1046
RandomTypoDestroyOperation         814
RandomWordRemoveOperation          409
CapitalsDestroyer                  168
SimpleQuestionDestroyer            138
Name: count, dtype: int64

Répartition des sources :
source
huggingface_multilingual_gec    65773
Name: count, dtype: int64


In [26]:
from pathlib import Path

# Créer le dossier de destination
dossier = Path("data/processed")
dossier.mkdir(parents=True, exist_ok=True)

# Sauvegarder les deux formats
fichier_csv = dossier / "corpus_nettoye.csv"
fichier_parquet = dossier / "corpus_nettoye.parquet"

corpus.to_csv(
    fichier_csv,
    index=False,
    encoding="utf-8"
)

corpus.to_parquet(
    fichier_parquet,
    index=False
)

# Vérifier les sauvegardes
print("CSV créé :", fichier_csv.resolve(), fichier_csv.exists())
print("Parquet créé :", fichier_parquet.resolve(), fichier_parquet.exists())

print("\nTaille du corpus :", len(corpus))

CSV créé : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data/processed/corpus_nettoye.csv True
Parquet créé : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data/processed/corpus_nettoye.parquet True

Taille du corpus : 65773


In [28]:
from pathlib import Path

dossier = Path("../data/raw")

print("Fichiers présents dans data/raw :\n")

for fichier in dossier.iterdir():
    if fichier.is_file():
        print(f"{fichier.name} — {fichier.stat().st_size / 1024:.2f} Ko")

Fichiers présents dans data/raw :

paires_5000.csv — 386.69 Ko
multilingual_gec_fr.parquet — 5416.37 Ko
paires.csv — 8.22 Ko
multilingual_gec_fr.csv — 12950.78 Ko


In [31]:
import pandas as pd
from pathlib import Path

# Chemins des deux fichiers
fichier_100 = Path("../data/raw/paires.csv")
fichier_5000 = Path("../data/raw/paires_5000.csv")

# Chargement
claude_100 = pd.read_csv(fichier_100)
claude_5000 = pd.read_csv(fichier_5000)

print("CORPUS CLAUDE 100")
print("Dimensions :", claude_100.shape)
print("Colonnes :", claude_100.columns.tolist())
display(claude_100.head(5))

print("\nCORPUS CLAUDE 5000")
print("Dimensions :", claude_5000.shape)
print("Colonnes :", claude_5000.columns.tolist())
display(claude_5000.head(5))

CORPUS CLAUDE 100
Dimensions : (100, 2)
Colonnes : ['incorrect', 'correct']


,incorrect,correct
0,Le chat dort sur le canape.,Le chat dort sur le canapé.
1,Les enfants joue dans le jardin.,Les enfants jouent dans le jardin.
2,Elle a mangé une pomme rouge et sucré.,Elle a mangé une pomme rouge et sucrée.
3,Nous allons a la plage demain.,Nous allons à la plage demain.
4,Il fait tres chaud aujourd'hui.,Il fait très chaud aujourd'hui.



CORPUS CLAUDE 5000
Dimensions : (5000, 2)
Colonnes : ['incorrect', 'correct']


,incorrect,correct
0,Le jolis voisin se lave chaque matin.,Le joli voisin se lave chaque matin.
1,Des garçons tranquilles ont dessine le bateau ...,Des garçons tranquilles ont dessiné le bateau ...
2,Il y a une mer à le vilage.,Il y a une mer au village.
3,Fatou nageront dans le piscine.,Fatou nagera dans la piscine.
4,Cinq institutrices ce promenaient toujours dan...,Cinq institutrices se promenaient toujours dan...


In [32]:
print("VALEURS MANQUANTES")

print("\nClaude 100 :")
print(claude_100.isna().sum())

print("\nClaude 5000 :")
print(claude_5000.isna().sum())

print("\nDOUBLONS")

print(
    "Claude 100 :",
    claude_100.duplicated().sum()
)

print(
    "Claude 5000 :",
    claude_5000.duplicated().sum()
)

VALEURS MANQUANTES

Claude 100 :
incorrect    0
correct      0
dtype: int64

Claude 5000 :
incorrect    0
correct      0
dtype: int64

DOUBLONS
Claude 100 : 0
Claude 5000 : 0


In [33]:
def verifier_corpus(nom, data):
    print(f"\n===== {nom} =====")

    # Vérifier les paires identiques
    identiques = (
        data["incorrect"].str.strip()
        == data["correct"].str.strip()
    ).sum()

    # Vérifier les doublons de paires
    doublons = data.duplicated(
        subset=["incorrect", "correct"]
    ).sum()

    # Vérifier les phrases vides
    incorrectes_vides = (
        data["incorrect"].str.strip() == ""
    ).sum()

    correctes_vides = (
        data["correct"].str.strip() == ""
    ).sum()

    print("Paires identiques :", identiques)
    print("Paires dupliquées :", doublons)
    print("Phrases incorrectes vides :", incorrectes_vides)
    print("Phrases correctes vides :", correctes_vides)


verifier_corpus("CLAUDE 100", claude_100)
verifier_corpus("CLAUDE 5000", claude_5000)


===== CLAUDE 100 =====
Paires identiques : 0
Paires dupliquées : 0
Phrases incorrectes vides : 0
Phrases correctes vides : 0

===== CLAUDE 5000 =====
Paires identiques : 0
Paires dupliquées : 0
Phrases incorrectes vides : 0
Phrases correctes vides : 0


In [34]:
# Construire des ensembles de paires
paires_100 = set(
    zip(claude_100["incorrect"], claude_100["correct"])
)

paires_5000 = set(
    zip(claude_5000["incorrect"], claude_5000["correct"])
)

paires_hf = set(
    zip(corpus["incorrect"], corpus["correct"])
)

# Comparer les trois corpus
print("Chevauchements entre Claude 100 et Claude 5000 :",
      len(paires_100 & paires_5000))

print("Chevauchements entre Claude 100 et Hugging Face :",
      len(paires_100 & paires_hf))

print("Chevauchements entre Claude 5000 et Hugging Face :",
      len(paires_5000 & paires_hf))

Chevauchements entre Claude 100 et Claude 5000 : 0
Chevauchements entre Claude 100 et Hugging Face : 0
Chevauchements entre Claude 5000 et Hugging Face : 0


In [35]:
import unicodedata

# Préparer le corpus Claude de 100 paires
claude_100_prepare = claude_100.copy()
claude_100_prepare["source"] = "claude_100"

# Préparer le corpus Claude de 5000 paires
claude_5000_prepare = claude_5000.copy()
claude_5000_prepare["source"] = "claude_5000"

# Normaliser Unicode et retirer les espaces extérieurs
def normaliser_texte(texte):
    return unicodedata.normalize("NFC", str(texte)).strip()

for data in [claude_100_prepare, claude_5000_prepare]:
    data["incorrect"] = data["incorrect"].apply(normaliser_texte)
    data["correct"] = data["correct"].apply(normaliser_texte)

In [36]:
# Préparer le corpus Hugging Face
hf_prepare = corpus.copy()

# Ajouter les colonnes absentes aux corpus Claude
for data in [claude_100_prepare, claude_5000_prepare]:
    data["transformation"] = pd.NA
    data["sec_transformation"] = pd.NA

# Fusionner les trois corpus
corpus_unifie = pd.concat(
    [
        hf_prepare,
        claude_100_prepare,
        claude_5000_prepare
    ],
    ignore_index=True
)

# Harmoniser les textes de toutes les sources
corpus_unifie["incorrect"] = corpus_unifie["incorrect"].apply(normaliser_texte)
corpus_unifie["correct"] = corpus_unifie["correct"].apply(normaliser_texte)

# Vérifier les paires identiques
identiques = (
    corpus_unifie["incorrect"] == corpus_unifie["correct"]
).sum()

# Vérifier les doublons
doublons = corpus_unifie.duplicated(
    subset=["incorrect", "correct"]
).sum()

print("Nombre total de paires :", len(corpus_unifie))
print("Paires identiques :", identiques)
print("Doublons :", doublons)

print("\nRépartition par source :")
print(corpus_unifie["source"].value_counts())

print("\nValeurs manquantes :")
print(corpus_unifie.isna().sum())

Nombre total de paires : 70873
Paires identiques : 0
Doublons : 0

Répartition par source :
source
huggingface_multilingual_gec    65773
claude_5000                      5000
claude_100                        100
Name: count, dtype: int64

Valeurs manquantes :
incorrect                 0
correct                   0
transformation         5100
sec_transformation    61895
source                    0
dtype: int64


In [37]:
from pathlib import Path

# Créer le dossier de destination s'il n'existe pas
dossier = Path("data/processed")
dossier.mkdir(parents=True, exist_ok=True)

# Définir les chemins
fichier_csv = dossier / "corpus_unifie.csv"
fichier_parquet = dossier / "corpus_unifie.parquet"

# Enregistrer les fichiers
corpus_unifie.to_csv(
    fichier_csv,
    index=False,
    encoding="utf-8"
)

corpus_unifie.to_parquet(
    fichier_parquet,
    index=False
)

print("Enregistrement terminé !")
print("CSV :", fichier_csv.resolve())
print("Parquet :", fichier_parquet.resolve())

Enregistrement terminé !
CSV : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data/processed/corpus_unifie.csv
Parquet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data/processed/corpus_unifie.parquet


In [38]:
print("CSV existe :", fichier_csv.exists())
print("Parquet existe :", fichier_parquet.exists())

print("Dimensions du corpus :", corpus_unifie.shape)

CSV existe : True
Parquet existe : True
Dimensions du corpus : (70873, 5)


In [39]:
# Prélever 20 paires dans chaque source
echantillons = []

for source, groupe in corpus_unifie.groupby("source"):

    n = min(20, len(groupe))

    echantillon = groupe.sample(
        n=n,
        random_state=42
    )

    echantillons.append(echantillon)

# Fusionner les échantillons
echantillon_final = pd.concat(
    echantillons,
    ignore_index=True
)

# Afficher les colonnes utiles
display(
    echantillon_final[
        ["source", "incorrect", "correct", "transformation"]
    ]
)

,source,incorrect,correct,transformation
0,claude_100,Les touristes visite le musée toute la journée.,Les touristes visitent le musée toute la journée.,<NA>
1,claude_100,Les chiens aboie quand quelqu'un sonne à la po...,Les chiens aboient quand quelqu'un sonne à la ...,<NA>
2,claude_100,Je crois que ces livres sont a toi.,Je crois que ces livres sont à toi.,<NA>
3,claude_100,Le train arrive a huit heures du matin.,Le train arrive à huit heures du matin.,<NA>
4,claude_100,Ces enfants son très sage aujourd'hui.,Ces enfants sont très sages aujourd'hui.,<NA>
5,claude_100,Les pommes que j'ai acheté sont délicieuses.,Les pommes que j'ai achetées sont délicieuses.,<NA>
6,claude_100,Je ne sais pas si il viendra.,Je ne sais pas s'il viendra.,<NA>
7,claude_100,Nous avons aperçu un renard pres de la foret.,Nous avons aperçu un renard près de la forêt.,<NA>
8,claude_100,Ou est ton frère ?,Où est ton frère ?,<NA>
9,claude_100,Le chat dort sur le canape.,Le chat dort sur le canapé.,<NA>


In [40]:
print("Nombre total de paires :", len(echantillon_final))

print("\nRépartition par source :")
print(echantillon_final["source"].value_counts())


Nombre total de paires : 60

Répartition par source :
source
claude_100                      20
claude_5000                     20
huggingface_multilingual_gec    20
Name: count, dtype: int64


In [41]:
for source in echantillon_final["source"].unique():

    print("\n" + "=" * 90)
    print("SOURCE :", source)
    print("=" * 90)

    groupe = echantillon_final[
        echantillon_final["source"] == source
    ]

    for numero, (_, ligne) in enumerate(groupe.iterrows(), start=1):

        print(f"\nPaire {numero}")
        print("INCORRECT :", ligne["incorrect"])
        print("CORRECT   :", ligne["correct"])

        if pd.notna(ligne["transformation"]):
            print("TRANSFORMATION :", ligne["transformation"])


SOURCE : claude_100

Paire 1
INCORRECT : Les touristes visite le musée toute la journée.
CORRECT   : Les touristes visitent le musée toute la journée.

Paire 2
INCORRECT : Les chiens aboie quand quelqu'un sonne à la porte.
CORRECT   : Les chiens aboient quand quelqu'un sonne à la porte.

Paire 3
INCORRECT : Je crois que ces livres sont a toi.
CORRECT   : Je crois que ces livres sont à toi.

Paire 4
INCORRECT : Le train arrive a huit heures du matin.
CORRECT   : Le train arrive à huit heures du matin.

Paire 5
INCORRECT : Ces enfants son très sage aujourd'hui.
CORRECT   : Ces enfants sont très sages aujourd'hui.

Paire 6
INCORRECT : Les pommes que j'ai acheté sont délicieuses.
CORRECT   : Les pommes que j'ai achetées sont délicieuses.

Paire 7
INCORRECT : Je ne sais pas si il viendra.
CORRECT   : Je ne sais pas s'il viendra.

Paire 8
INCORRECT : Nous avons aperçu un renard pres de la foret.
CORRECT   : Nous avons aperçu un renard près de la forêt.

Paire 9
INCORRECT : Ou est ton frère 

In [42]:
import re

def phrase_suspecte(texte):
    texte = str(texte).strip()

    # Texte vide
    if not texte:
        return True

    # Terminaison manifestement incomplète
    if texte.endswith(("(", "[", "{", "«", "'")):
        return True

    # Parenthèses, crochets ou accolades non équilibrés
    for ouverture, fermeture in [
        ("(", ")"),
        ("[", "]"),
        ("{", "}")
    ]:
        if texte.count(ouverture) > texte.count(fermeture):
            return True

    # Guillemets français non équilibrés
    if texte.count("«") > texte.count("»"):
        return True

    return False

In [43]:
# Ne pas modifier le corpus original
corpus_controle = corpus_unifie.copy()

# Identifier les phrases suspectes
corpus_controle["suspect_incorrect"] = (
    corpus_controle["incorrect"].apply(phrase_suspecte)
)

corpus_controle["suspect_correct"] = (
    corpus_controle["correct"].apply(phrase_suspecte)
)

# Une paire est suspecte si l'une des deux phrases l'est
corpus_controle["a_verifier"] = (
    corpus_controle["suspect_incorrect"] |
    corpus_controle["suspect_correct"]
)

# Afficher le bilan
print("Nombre total de paires :", len(corpus_controle))

print(
    "Paires suspectes :",
    corpus_controle["a_verifier"].sum()
)

print("\nRépartition par source :")

print(
    corpus_controle.groupby("source")["a_verifier"].sum()
)

Nombre total de paires : 70873
Paires suspectes : 3433

Répartition par source :
source
claude_100                         0
claude_5000                        0
huggingface_multilingual_gec    3433
Name: a_verifier, dtype: int64


In [44]:
display(
    corpus_controle[
        corpus_controle["a_verifier"]
    ][
        [
            "source",
            "incorrect",
            "correct",
            "suspect_incorrect",
            "suspect_correct"
        ]
    ].head(30)
)

,source,incorrect,correct,suspect_incorrect,suspect_correct
1,huggingface_multilingual_gec,"En 1933-1934, elle plaide auprès du président ...","En 1933-1934, elle plaide auprès du président ...",True,True
4,huggingface_multilingual_gec,"Le fatimide général envahit l'émirat, vainc un...","Le général fatimide envahit l'émirat, vainc un...",True,True
34,huggingface_multilingual_gec,"L'situation intérieure de l'double monarchie, ...",La situation intérieure de la double monarchie...,True,True
36,huggingface_multilingual_gec,Selon les syndicats policiers « Il n'suffit pa...,Selon les syndicats policiers « Il ne suffit p...,True,True
48,huggingface_multilingual_gec,Il semble que Cheissoux ait été une étape impo...,Il semble que Cheissoux ait été une étape impo...,False,True
49,huggingface_multilingual_gec,Notez que : 'Rheims' est l'orthographes ancien...,Notez que : 'Rheims' est l'orthographe ancien ...,True,True
54,huggingface_multilingual_gec,La Grande Guerre va donner l’occasion du vérif...,La Grande Guerre va donner l’occasion de vérif...,True,True
87,huggingface_multilingual_gec,"La yérou à yer postérieur (capitale Ꙑ, minuscu...","Le yérou à yer postérieur (capitale Ꙑ, minuscu...",True,True
107,huggingface_multilingual_gec,Elle le défned avec uen tournée de 18 dates à ...,Elle le défend avec une tournée de 18 dates à ...,True,True
141,huggingface_multilingual_gec,Selon les syndicats policiers « Il ne suffit d...,Selon les syndicats policiers « Il ne suffit p...,True,True


In [45]:
def raisons_suspicion(texte):
    texte = str(texte).strip()
    raisons = []

    if not texte:
        raisons.append("Texte vide")

    if texte.endswith(("(", "[", "{")):
        raisons.append("Ponctuation ouvrante en fin de phrase")

    if texte.endswith("«"):
        raisons.append("Guillemet ouvrant en fin de phrase")

    if texte.endswith("'"):
        raisons.append("Apostrophe en fin de phrase")

    if texte.endswith("’"):
        raisons.append("Apostrophe typographique en fin de phrase")

    for ouverture, fermeture in [
        ("(", ")"),
        ("[", "]"),
        ("{", "}")
    ]:
        if texte.count(ouverture) > texte.count(fermeture):
            raisons.append(
                f"Parenthèse ou crochet non fermé : {ouverture}"
            )

    if texte.count("«") > texte.count("»"):
        raisons.append("Guillemet français non fermé")

    return raisons


# Examiner les raisons pour chaque phrase
corpus_controle["raisons_incorrect"] = (
    corpus_controle["incorrect"].apply(raisons_suspicion)
)

corpus_controle["raisons_correct"] = (
    corpus_controle["correct"].apply(raisons_suspicion)
)

# Afficher les fréquences des différentes raisons
from collections import Counter

frequences = Counter()

for raisons in corpus_controle["raisons_incorrect"]:
    frequences.update(raisons)

for raisons in corpus_controle["raisons_correct"]:
    frequences.update(raisons)

print("Fréquence des raisons de suspicion :")

for raison, nombre in frequences.most_common():
    print(f"{raison} : {nombre}")

Fréquence des raisons de suspicion :
Apostrophe en fin de phrase : 4742
Parenthèse ou crochet non fermé : ( : 1306
Ponctuation ouvrante en fin de phrase : 1038
Apostrophe typographique en fin de phrase : 805
Guillemet français non fermé : 680
Guillemet ouvrant en fin de phrase : 320


In [46]:
suspects = corpus_controle[
    corpus_controle["a_verifier"]
].copy()

for _, ligne in suspects.head(30).iterrows():

    print("\n" + "-" * 80)
    print("INCORRECT :", repr(ligne["incorrect"]))
    print("CORRECT   :", repr(ligne["correct"]))
    print("Raison incorrect :", ligne["raisons_incorrect"])
    print("Raison correct   :", ligne["raisons_correct"])


--------------------------------------------------------------------------------
INCORRECT : "En 1933-1934, elle plaide auprès du président pour un adhésion des États-Unis à l'"
CORRECT   : "En 1933-1934, elle plaide auprès du président pour une adhésion des États-Unis à l'"
Raison incorrect : ['Apostrophe en fin de phrase']
Raison correct   : ['Apostrophe en fin de phrase']

--------------------------------------------------------------------------------
INCORRECT : "Le fatimide général envahit l'émirat, vainc une byzantine force dirigée par le dux d'"
CORRECT   : "Le général fatimide envahit l'émirat, vainc une force byzantine dirigée par le dux d'"
Raison incorrect : ['Apostrophe en fin de phrase']
Raison correct   : ['Apostrophe en fin de phrase']

--------------------------------------------------------------------------------
INCORRECT : "L'situation intérieure de l'double monarchie, les relations tendues entre l'"
CORRECT   : "La situation intérieure de la double monarchie, les

In [47]:
import re

def detecter_troncature(texte):
    texte = str(texte).strip()
    raisons = []

    # Une phrase ne doit pas être vide
    if not texte:
        return ["Texte vide"]

    # Terminaisons typiques d'une élision coupée
    if re.search(
        r"\b(?:l|d|j|n|c|s|m|t|qu|jusqu|lorsqu|puisqu|quoiqu|quelqu|presqu)[’']$",
        texte,
        flags=re.IGNORECASE
    ):
        raisons.append("Élision incomplète en fin de phrase")

    # Parenthèses, crochets et accolades non fermés
    for ouverture, fermeture in [
        ("(", ")"),
        ("[", "]"),
        ("{", "}")
    ]:
        if texte.count(ouverture) > texte.count(fermeture):
            raisons.append("Ponctuation ouvrante non fermée")

    # Guillemets français non équilibrés
    if texte.count("«") > texte.count("»"):
        raisons.append("Guillemet français non fermé")

    return raisons


# Examiner les deux phrases
corpus_controle["raisons_incorrect_v2"] = (
    corpus_controle["incorrect"].apply(detecter_troncature)
)

corpus_controle["raisons_correct_v2"] = (
    corpus_controle["correct"].apply(detecter_troncature)
)

# Une paire est suspecte si au moins une phrase est tronquée
corpus_controle["troncature_v2"] = (
    corpus_controle["raisons_incorrect_v2"].apply(bool)
    |
    corpus_controle["raisons_correct_v2"].apply(bool)
)

print(
    "Nombre de paires suspectes après amélioration :",
    corpus_controle["troncature_v2"].sum()
)

print("\nRépartition par source :")

print(
    corpus_controle.groupby("source")["troncature_v2"].sum()
)

Nombre de paires suspectes après amélioration : 3810

Répartition par source :
source
claude_100                         0
claude_5000                        0
huggingface_multilingual_gec    3810
Name: troncature_v2, dtype: int64


In [48]:
display(
    corpus_controle[
        corpus_controle["troncature_v2"]
    ][
        [
            "source",
            "incorrect",
            "correct",
            "raisons_incorrect_v2",
            "raisons_correct_v2"
        ]
    ].head(40)
)

,source,incorrect,correct,raisons_incorrect_v2,raisons_correct_v2
1,huggingface_multilingual_gec,"En 1933-1934, elle plaide auprès du président ...","En 1933-1934, elle plaide auprès du président ...",[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
4,huggingface_multilingual_gec,"Le fatimide général envahit l'émirat, vainc un...","Le général fatimide envahit l'émirat, vainc un...",[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
8,huggingface_multilingual_gec,"Le propriétaires du la ferme Walserhof, la fam...","Les propriétaires de la ferme Walserhof, la fa...",[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
34,huggingface_multilingual_gec,"L'situation intérieure de l'double monarchie, ...",La situation intérieure de la double monarchie...,[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
36,huggingface_multilingual_gec,Selon les syndicats policiers « Il n'suffit pa...,Selon les syndicats policiers « Il ne suffit p...,[Guillemet français non fermé],[Guillemet français non fermé]
48,huggingface_multilingual_gec,Il semble que Cheissoux ait été une étape impo...,Il semble que Cheissoux ait été une étape impo...,[],[Élision incomplète en fin de phrase]
50,huggingface_multilingual_gec,"En philosophie, Vivès s’en prenaient violemmen...","En philosophie, Vivès s’en prit violemment à l...",[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
54,huggingface_multilingual_gec,La Grande Guerre va donner l’occasion du vérif...,La Grande Guerre va donner l’occasion de vérif...,[Ponctuation ouvrante non fermée],[Ponctuation ouvrante non fermée]
87,huggingface_multilingual_gec,"La yérou à yer postérieur (capitale Ꙑ, minuscu...","Le yérou à yer postérieur (capitale Ꙑ, minuscu...",[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]
107,huggingface_multilingual_gec,Elle le défned avec uen tournée de 18 dates à ...,Elle le défend avec une tournée de 18 dates à ...,[Élision incomplète en fin de phrase],[Élision incomplète en fin de phrase]


In [49]:
from collections import Counter

# Compter les raisons détectées dans les phrases incorrectes
raisons_incorrectes = Counter(
    raison
    for liste in corpus_controle["raisons_incorrect_v2"]
    for raison in liste
)

# Compter les raisons détectées dans les phrases correctes
raisons_correctes = Counter(
    raison
    for liste in corpus_controle["raisons_correct_v2"]
    for raison in liste
)

print("Raisons détectées dans les phrases incorrectes :")
for raison, nombre in raisons_incorrectes.items():
    print(f"- {raison} : {nombre}")

print("\nRaisons détectées dans les phrases correctes :")
for raison, nombre in raisons_correctes.items():
    print(f"- {raison} : {nombre}")

Raisons détectées dans les phrases incorrectes :
- Élision incomplète en fin de phrase : 2683
- Guillemet français non fermé : 340
- Ponctuation ouvrante non fermée : 630

Raisons détectées dans les phrases correctes :
- Élision incomplète en fin de phrase : 2843
- Guillemet français non fermé : 340
- Ponctuation ouvrante non fermée : 676


In [50]:
suspects = corpus_controle[corpus_controle["troncature_v2"]].copy()

for _, ligne in suspects.head(30).iterrows():
    print("-" * 80)
    print("INCORRECT :", repr(ligne["incorrect"][-100:]))
    print("CORRECT   :", repr(ligne["correct"][-100:]))
    print("RAISONS incorrect :", ligne["raisons_incorrect_v2"])
    print("RAISONS correct   :", ligne["raisons_correct_v2"])

--------------------------------------------------------------------------------
INCORRECT : "En 1933-1934, elle plaide auprès du président pour un adhésion des États-Unis à l'"
CORRECT   : "En 1933-1934, elle plaide auprès du président pour une adhésion des États-Unis à l'"
RAISONS incorrect : ['Élision incomplète en fin de phrase']
RAISONS correct   : ['Élision incomplète en fin de phrase']
--------------------------------------------------------------------------------
INCORRECT : "Le fatimide général envahit l'émirat, vainc une byzantine force dirigée par le dux d'"
CORRECT   : "Le général fatimide envahit l'émirat, vainc une force byzantine dirigée par le dux d'"
RAISONS incorrect : ['Élision incomplète en fin de phrase']
RAISONS correct   : ['Élision incomplète en fin de phrase']
--------------------------------------------------------------------------------
INCORRECT : ' famille Walser, reçurent, entre autres, après la guerre une lettre du remerciement du la part du l’'
CORRECT

In [51]:
from pathlib import Path

# Copier les paires suspectes
paires_suspectes = corpus_controle[
    corpus_controle["troncature_v2"]
].copy()

# Créer le dossier si nécessaire
dossier = Path("data/processed")
dossier.mkdir(parents=True, exist_ok=True)

# Sauvegarder les paires suspectes
fichier_suspects = dossier / "paires_suspectes_troncature.csv"

paires_suspectes.to_csv(
    fichier_suspects,
    index=False,
    encoding="utf-8"
)

print("Nombre de paires sauvegardées :", len(paires_suspectes))
print("Fichier créé :", fichier_suspects.resolve())
print("Fichier existant :", fichier_suspects.exists())

Nombre de paires sauvegardées : 3810
Fichier créé : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data/processed/paires_suspectes_troncature.csv
Fichier existant : True


In [52]:
# Créer une nouvelle copie sans les paires suspectes
corpus_final = corpus_controle.loc[
    ~corpus_controle["troncature_v2"],
    corpus_unifie.columns
].copy()

# Vérifications générales
print("Nombre de paires avant nettoyage :", len(corpus_unifie))
print("Nombre de paires retirées :", len(corpus_unifie) - len(corpus_final))
print("Nombre de paires restantes :", len(corpus_final))

print("\n--- Vérification de la qualité ---")

print(
    "Valeurs manquantes :",
    corpus_final[["incorrect", "correct"]].isna().sum().sum()
)

print(
    "Paires identiques :",
    (corpus_final["incorrect"] == corpus_final["correct"]).sum()
)

print(
    "Doublons :",
    corpus_final.duplicated(
        subset=["incorrect", "correct"]
    ).sum()
)

print(
    "Paires encore signalées par le détecteur v2 :",
    corpus_controle.loc[corpus_final.index, "troncature_v2"].sum()
)

print("\n--- Répartition par source ---")

print(corpus_final["source"].value_counts())

Nombre de paires avant nettoyage : 70873
Nombre de paires retirées : 3810
Nombre de paires restantes : 67063

--- Vérification de la qualité ---
Valeurs manquantes : 0
Paires identiques : 0
Doublons : 0
Paires encore signalées par le détecteur v2 : 0

--- Répartition par source ---
source
huggingface_multilingual_gec    61963
claude_5000                      5000
claude_100                        100
Name: count, dtype: int64


In [53]:
from pathlib import Path
import pandas as pd

# Définir les chemins de sauvegarde
dossier = Path("data/processed")

fichier_csv = dossier / "corpus_final.csv"
fichier_parquet = dossier / "corpus_final.parquet"

# Sauvegarder le corpus final
corpus_final.to_csv(
    fichier_csv,
    index=False,
    encoding="utf-8"
)

corpus_final.to_parquet(
    fichier_parquet,
    index=False
)

# Vérifier les fichiers enregistrés en les rechargeant
verification_csv = pd.read_csv(fichier_csv)
verification_parquet = pd.read_parquet(fichier_parquet)

print("Sauvegarde terminée.")

print("\nCSV :")
print("Fichier existant :", fichier_csv.exists())
print("Dimensions :", verification_csv.shape)

print("\nParquet :")
print("Fichier existant :", fichier_parquet.exists())
print("Dimensions :", verification_parquet.shape)

print("\nColonnes :", list(corpus_final.columns))

Sauvegarde terminée.

CSV :
Fichier existant : True
Dimensions : (67063, 5)

Parquet :
Fichier existant : True
Dimensions : (67063, 5)

Colonnes : ['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']


In [54]:
from pathlib import Path

# Afficher le dossier de travail actuel
dossier_actuel = Path.cwd()

print("Dossier de travail actuel :")
print(dossier_actuel)

print("\nDossiers data détectés :")

for dossier in dossier_actuel.parent.rglob("data"):
    if dossier.is_dir():
        print(dossier.resolve())

Dossier de travail actuel :
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks

Dossiers data détectés :
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/notebooks/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/.venv/lib/python3.12/site-packages/numpy/typing/tests/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/.venv/lib/python3.12/site-packages/numpy/_core/tests/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/.venv/lib/python3.12/site-packages/numpy/random/tests/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/.venv/lib/python3.12/site-packages/numpy/lib/tests/data
/home/nana-marc/Documents/coursBazongo/projetdeep/COFI/.venv/lib/python3.12/site-packages/pyarrow/tests/data


In [55]:
from pathlib import Path

dossier_data = Path("data")

print("Contenu de notebooks/data :\n")

if dossier_data.exists():
    for fichier in dossier_data.rglob("*"):
        if fichier.is_file():
            print(
                "Fichier :", fichier.relative_to(dossier_data),
                "| Taille :", round(fichier.stat().st_size / 1024, 2), "Ko"
            )
else:
    print("Ce dossier n'existe pas.")

Contenu de notebooks/data :

Fichier : processed/corpus_nettoye.csv | Taille : 15837.38 Ko
Fichier : processed/corpus_unifie.csv | Taille : 16297.0 Ko
Fichier : processed/corpus_unifie.parquet | Taille : 5139.99 Ko
Fichier : processed/corpus_final.csv | Taille : 15405.15 Ko
Fichier : processed/paires_suspectes_troncature.csv | Taille : 1582.01 Ko
Fichier : processed/corpus_final.parquet | Taille : 4859.89 Ko
Fichier : processed/corpus_nettoye.parquet | Taille : 4922.95 Ko


In [56]:
from pathlib import Path

dossiers = {
    "Dossier principal": Path("../data/processed"),
    "Dossier secondaire": Path("data/processed")
}

for nom, dossier in dossiers.items():
    print(f"\n--- {nom} ---")

    for fichier in sorted(dossier.glob("*")):
        if fichier.is_file():
            taille = round(fichier.stat().st_size / 1024, 2)
            print(f"{fichier.name} : {taille} Ko")


--- Dossier principal ---
corpus_final.csv : 15405.15 Ko
corpus_final.parquet : 4859.89 Ko
corpus_nettoye.csv : 15837.38 Ko
corpus_nettoye.parquet : 4922.95 Ko
corpus_unifie.csv : 16297.0 Ko
corpus_unifie.parquet : 5139.99 Ko
paires_suspectes_troncature.csv : 1582.01 Ko

--- Dossier secondaire ---
corpus_final.csv : 15405.15 Ko
corpus_final.parquet : 4859.89 Ko
corpus_nettoye.csv : 15837.38 Ko
corpus_nettoye.parquet : 4922.95 Ko
corpus_unifie.csv : 16297.0 Ko
corpus_unifie.parquet : 5139.99 Ko
paires_suspectes_troncature.csv : 1582.01 Ko


In [57]:
import shutil
from pathlib import Path

dossier_secondaire = Path("data")

if dossier_secondaire.exists():
    shutil.rmtree(dossier_secondaire)
    print("Dossier notebooks/data supprimé avec succès.")
else:
    print("Le dossier notebooks/data n'existe déjà plus.")

Dossier notebooks/data supprimé avec succès.


In [58]:
from pathlib import Path

print("Dossier principal existe :", Path("../data").exists())
print("Dossier secondaire existe :", Path("data").exists())

print("\nFichiers conservés dans le dossier principal :")

for fichier in sorted(Path("../data/processed").glob("*")):
    print(fichier.name)

Dossier principal existe : True
Dossier secondaire existe : False

Fichiers conservés dans le dossier principal :
corpus_final.csv
corpus_final.parquet
corpus_nettoye.csv
corpus_nettoye.parquet
corpus_unifie.csv
corpus_unifie.parquet
paires_suspectes_troncature.csv
